In [0]:
# 02_preprocess_data

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

catalog = "workspace"
schema = "stroke_prediction"

raw_table = f"{catalog}.{schema}.stroke_raw"
clean_table = f"{catalog}.{schema}.stroke_clean"
train_table = f"{catalog}.{schema}.stroke_train"
test_table = f"{catalog}.{schema}.stroke_test"
split_summary_table = f"{catalog}.{schema}.stroke_split_summary"

random_state = 42

stroke = spark.table(raw_table).toPandas()

stroke_clean = (
    stroke
    .loc[
        ~stroke["work_type"].isin(["Never_worked", "children"])
        & stroke["gender"].ne("Other")
    ]
    .drop(columns="id")
    .replace({"smoking_status": {"Unknown": np.nan}, "bmi": {"N/A": np.nan}})
    .reset_index(drop=True)
)

if stroke_clean["stroke"].isna().any():
    raise ValueError("Missing stroke outcomes detected.")

if not set(stroke_clean["stroke"].unique()).issubset({0, 1}):
    raise ValueError("Invalid stroke outcomes detected.")

spark.createDataFrame(stroke_clean).write.mode("overwrite").saveAsTable(clean_table)

X = stroke_clean.drop(columns="stroke")
y = stroke_clean["stroke"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=random_state
)

train_data = X_train.assign(stroke=y_train).reset_index(drop=True)
test_data = X_test.assign(stroke=y_test).reset_index(drop=True)

spark.createDataFrame(train_data).write.mode("overwrite").saveAsTable(train_table)
spark.createDataFrame(test_data).write.mode("overwrite").saveAsTable(test_table)

split_summary = pd.DataFrame({
    "dataset": ["Complete", "Training", "Test"],
    "patients": [
        len(stroke_clean),
        len(train_data),
        len(test_data)
    ],
    "stroke_cases": [
        int(stroke_clean["stroke"].sum()),
        int(train_data["stroke"].sum()),
        int(test_data["stroke"].sum())
    ],
    "stroke_prevalence": [
        float(stroke_clean["stroke"].mean()),
        float(train_data["stroke"].mean()),
        float(test_data["stroke"].mean())
    ]
})

spark.createDataFrame(split_summary).write.mode("overwrite").saveAsTable(
    split_summary_table
)

display(split_summary)
